In [1]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA를 사용할 수 없습니다.")

PyTorch 버전: 2.5.1+cu118
CUDA 사용 가능 여부: True
GPU 이름: NVIDIA GeForce RTX 4060 Ti


In [9]:
pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 23.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


실제 CSV 데이터를 불러와 전처리 및 피처 엔지니어링 수행

Optuna로 XGBRegressor 하이퍼파라미터 자동 탐색 (500회 시도)

최적 파라미터로 최종 모델 학습

학습 데이터 기준 RMSE, MAE, R² 성능 지표 출력

### XGB

In [23]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score
import optuna
import random
from collections import Counter
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

# ------------------------
# [1] 데이터 로드 및 전처리
# ------------------------
print("\n데이터 로드 중...")
train = pd.read_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/상준's 데이터/2017_2023_정규_플옵_재구성완료.csv")
valid = pd.read_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/2024_25_파생변수_포함_WIN_PROB_정제최종.csv")

train["TEAM_NAME"] = train["TEAM_NAME"].str.lower()
valid["TEAM_NAME"] = valid["TEAM_NAME"].str.lower()

base_features = [
    'FGM', 'PLUS_MINUS_x', 'FG_PCT_x', 'AST_x', 'FG3M', 'DREB', 'REB_x'
]

def create_features(df):
    df['GAME_NUM'] = df.groupby(['SEASON', 'TEAM_NAME']).cumcount() + 1
    df['SEASON_PROGRESS'] = df['GAME_NUM'] / 82
    df['OPP_PTS_DIFF'] = df['PTS_x'] - df['OPP_PTS']
    df['OPP_PTS_ALLOWED_DIFF'] = df['OPP_PTS'] - df['PTS_x']
    df['FG_EFFICIENCY'] = df['FGM'] / df['FGA']
    df['FG3_EFFICIENCY'] = df['FG3M'] / df['FG3A']
    df['SCORING_PACE'] = df['PTS_x'] / df['GAME_NUM']
    df['DEFENSE_RATING'] = df['OPP_PTS'] / df['GAME_NUM']
    df['HOME_ADVANTAGE'] = df['HOME'].astype(int)
    df['RECENT_PTS_AVG'] = df.groupby('TEAM_NAME')['PTS_x'].rolling(5).mean().reset_index(0, drop=True)
    df['RECENT_OPP_PTS_AVG'] = df.groupby('TEAM_NAME')['OPP_PTS'].rolling(5).mean().reset_index(0, drop=True)
    df['RECENT_WIN_RATE'] = df.groupby('TEAM_NAME')['WIN'].rolling(5).mean().reset_index(0, drop=True)
    return df

train = create_features(train)
valid = create_features(valid)

features = base_features + [
    'SEASON_PROGRESS', 'OPP_PTS_DIFF', 'OPP_PTS_ALLOWED_DIFF',
    'HOME_ADVANTAGE', 'FG_EFFICIENCY', 'FG3_EFFICIENCY',
    'SCORING_PACE', 'DEFENSE_RATING', 'RECENT_PTS_AVG',
    'RECENT_OPP_PTS_AVG', 'RECENT_WIN_RATE'
]

train = train.dropna(subset=features + ['PTS_x']).reset_index(drop=True)
valid = valid.dropna(subset=features + ['PTS_x']).reset_index(drop=True)

scaler = RobustScaler()
train[features] = scaler.fit_transform(train[features])
valid[features] = scaler.transform(valid[features])

# ------------------------
# [2] Optuna + XGBoost (GPU) 모델 학습 및 검증
# ------------------------
def objective_xgb(trial):
    model = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 300, 800),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 0.0, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.0, 3.0),
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        random_state=42,
        verbosity=0
    )
    score = cross_val_score(model, train[features], train['PTS_x'], scoring="neg_root_mean_squared_error", cv=5)
    return -score.mean()

print("\nOptuna를 이용한 XGBoost GPU 모델 하이퍼파라미터 탐색 중...")
study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=500)

print("\n✅ Optuna가 선택한 최적 XGBoost GPU 모델 파라미터:")
print(study_xgb.best_params)

best_model = XGBRegressor(
    **study_xgb.best_params,
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    random_state=42,
    verbosity=0
)
best_model.fit(
    train[features], train['PTS_x'],
    eval_set=[(valid[features], valid['PTS_x'])],
    early_stopping_rounds=20,
    verbose=False
)

print("\n✅ XGBoost GPU 모델 성능 평가")
y_pred_train = best_model.predict(train[features])
y_pred_valid = best_model.predict(valid[features])

print("\n[Train Performance]")
print(f"RMSE: {np.sqrt(mean_squared_error(train['PTS_x'], y_pred_train)):.4f}")
print(f"MAE: {mean_absolute_error(train['PTS_x'], y_pred_train):.4f}")
print(f"R²: {r2_score(train['PTS_x'], y_pred_train):.4f}")

print("\n[Valid Performance]")
print(f"RMSE: {np.sqrt(mean_squared_error(valid['PTS_x'], y_pred_valid)):.4f}")
print(f"MAE: {mean_absolute_error(valid['PTS_x'], y_pred_valid):.4f}")
print(f"R²: {r2_score(valid['PTS_x'], y_pred_valid):.4f}")

# ------------------------
# [3] 시뮬레이션 설정값 정의 (noise_std 증가)
# ------------------------
SIMULATION_CONFIG = {
    'n_simulations': 1000,
    'noise_std': 4.5,  # 기존보다 증가
    'home_advantage': {
        'base': 3.0,
        'playoff': 1.5
    },
    'recent_games_weight': 0.7,
    'min_games': 10
}


# ------------------------
# [6] 4강 및 결승 진출팀 필터링
# ------------------------
print("\n4강 및 결승 진출팀 필터링 중...")

# 4강 팀 정의
semifinal_teams = {
    "east": ["indiana pacers", "new york knicks"],
    "west": ["oklahoma city thunder", "minnesota timberwolves"]
}

# 4강 데이터 필터링
valid_semifinals = valid[valid["TEAM_NAME"].isin([team for teams in semifinal_teams.values() for team in teams])].copy()

# 팀별 데이터 수 출력
print("\n4강 팀별 데이터 수:")
print(valid_semifinals["TEAM_NAME"].value_counts())

# 데이터 밸런싱 함수
def balance_team_data(df, team, min_games):
    team_data = df[df["TEAM_NAME"] == team]
    if len(team_data) < min_games:
        recent_data = team_data.sort_values('GAME_NUM', ascending=False).head(5)
        recent_avg = recent_data[features].mean()
        additional_data = pd.DataFrame([recent_avg] * (min_games - len(team_data)))
        additional_data["TEAM_NAME"] = team
        return pd.concat([df, additional_data], ignore_index=True)
    return df

# 4강 데이터 밸런싱
for conference in semifinal_teams:
    for team in semifinal_teams[conference]:
        valid_semifinals = balance_team_data(valid_semifinals, team, SIMULATION_CONFIG['min_games'])

# ------------------------
# [7] 4강 시뮬레이션
# ------------------------
print("\n4강 시뮬레이션 중...")

def add_noise_to_score(score, std=SIMULATION_CONFIG['noise_std']):
    return score + np.random.normal(0, std)

def get_home_advantage(game_num, is_playoff=True):
    base_advantage = SIMULATION_CONFIG['home_advantage']['base']
    if is_playoff:
        base_advantage += SIMULATION_CONFIG['home_advantage']['playoff']
    return base_advantage

def simulate_game(team1_data, team2_data, features, game_num, is_home=True):
    if len(team1_data) > 0 and len(team2_data) > 0:
        n1 = len(team1_data)
        n2 = len(team2_data)
        series_weight = 1.0 + (game_num * 0.1)
        weights1 = np.exp(-np.arange(n1) * (1 - SIMULATION_CONFIG['recent_games_weight'])) * series_weight
        weights1 = weights1 / weights1.sum()
        weights2 = np.exp(-np.arange(n2) * (1 - SIMULATION_CONFIG['recent_games_weight'])) * series_weight
        weights2 = weights2 / weights2.sum()
        t1_row = team1_data.sample(n=1, weights=weights1, random_state=np.random.randint(0, 10000))
        t2_row = team2_data.sample(n=1, weights=weights2, random_state=np.random.randint(0, 10000))
        home_advantage = get_home_advantage(game_num) if is_home else 0.0
        if game_num >= 4:
            home_advantage *= 1.2
        t1_score = add_noise_to_score(best_model.predict(t1_row[features])[0] + home_advantage)
        t2_score = add_noise_to_score(best_model.predict(t2_row[features])[0])
        t1_score = max(70, min(150, t1_score))
        t2_score = max(70, min(150, t2_score))
        return t1_score, t2_score
    return None, None

def run_semifinal_simulation(team1, team2, team1_data, team2_data, features):
    team1_wins = 0
    team2_wins = 0
    series_scores = []
    
    for game_num in range(7):
        if team1_wins >= 4 or team2_wins >= 4:
            break
            
        is_home = (game_num < 2) or (game_num >= 4 and game_num < 6)
        t1_score, t2_score = simulate_game(team1_data, team2_data, features, game_num, is_home)
        
        if t1_score is not None and t2_score is not None:
            if t1_score > t2_score:
                team1_wins += 1
                winner = team1
            else:
                team2_wins += 1
                winner = team2
            series_scores.append({
                "team1_score": t1_score,
                "team2_score": t2_score,
                "winner": winner,
                "is_home": is_home,
                "game_num": game_num + 1
            })
    
    return {
        "team1": team1,
        "team2": team2,
        "team1_wins": team1_wins,
        "team2_wins": team2_wins,
        "series_scores": series_scores
    }

# 4강 시뮬레이션 실행
semifinal_results = {}
conference_winners = {}

for conference, teams in semifinal_teams.items():
    print(f"\n{conference.upper()} 컨퍼런스 4강 시뮬레이션 중...")
    simulation_results = []
    for _ in tqdm(range(SIMULATION_CONFIG['n_simulations']), desc=f"{conference} 시뮬레이션"):
        result = run_semifinal_simulation(
            teams[0],
            teams[1],
            valid_semifinals[valid_semifinals["TEAM_NAME"] == teams[0]],
            valid_semifinals[valid_semifinals["TEAM_NAME"] == teams[1]],
            features
        )
        simulation_results.append(result)
    semifinal_results[conference] = simulation_results
    
    # 컨퍼런스 승자 결정
    team1_wins = sum(1 for res in simulation_results if res['team1_wins'] > res['team2_wins'])
    team2_wins = sum(1 for res in simulation_results if res['team2_wins'] > res['team1_wins'])
    conference_winners[conference] = teams[0] if team1_wins > team2_wins else teams[1]

# 결승 진출팀 결정
finals_teams = [conference_winners['west'], conference_winners['east']]
print(f"\n결승 진출팀: {finals_teams[0]} vs {finals_teams[1]}")

# 결승 데이터 필터링
valid_finals = valid[valid["TEAM_NAME"].isin(finals_teams)].copy()

# 결승 데이터 밸런싱
for team in finals_teams:
    valid_finals = balance_team_data(valid_finals, team, SIMULATION_CONFIG['min_games'])

# ------------------------
# [8] 결승전 시뮬레이션
# ------------------------
print("\n결승전 시뮬레이션 중...")
finals_simulation_results = []
for _ in tqdm(range(SIMULATION_CONFIG['n_simulations']), desc="결승전 시뮬레이션"):
    result = run_semifinal_simulation(
        finals_teams[0],
        finals_teams[1],
        valid_finals[valid_finals["TEAM_NAME"] == finals_teams[0]],
        valid_finals[valid_finals["TEAM_NAME"] == finals_teams[1]],
        features
    )
    finals_simulation_results.append(result)

# ------------------------
# [9] 결과 출력 및 저장
# ------------------------
print("\n=== NBA 4강 및 결승전 시뮬레이션 결과 ===")

# 4강 결과 출력
for conference, results in semifinal_results.items():
    print(f"\n{conference.upper()} 컨퍼런스 4강 결과:")
    team1, team2 = semifinal_teams[conference]
    
    team1_wins = sum(1 for res in results if res['team1_wins'] > res['team2_wins'])
    team2_wins = sum(1 for res in results if res['team2_wins'] > res['team1_wins'])
    team1_win_rate = team1_wins / SIMULATION_CONFIG['n_simulations'] * 100
    team2_win_rate = team2_wins / SIMULATION_CONFIG['n_simulations'] * 100
    
    print(f"\n승률:")
    print(f"{team1}: {team1_win_rate:.1f}%")
    print(f"{team2}: {team2_win_rate:.1f}%")
    
    # 평균 득점 계산
    team1_avg_score = np.mean([score['team1_score'] for res in results for score in res['series_scores']])
    team2_avg_score = np.mean([score['team2_score'] for res in results for score in res['series_scores']])
    team1_std_score = np.std([score['team1_score'] for res in results for score in res['series_scores']])
    team2_std_score = np.std([score['team2_score'] for res in results for score in res['series_scores']])
    
    print(f"\n평균 득점 (표준편차):")
    print(f"{team1}: {team1_avg_score:.1f} (±{team1_std_score:.1f})")
    print(f"{team2}: {team2_avg_score:.1f} (±{team2_std_score:.1f})")
    
    # 시리즈 스코어 분포
    series_scores = Counter([f"{res['team1_wins']}:{res['team2_wins']}" for res in results])
    print("\n시리즈 스코어 분포:")
    for score, count in series_scores.most_common():
        print(f"{score}: {count/SIMULATION_CONFIG['n_simulations']*100:.1f}%")
    
    # 가장 많이 나온 시리즈 스코어의 경기별 득점
    most_common_score = series_scores.most_common(1)[0][0]
    team1_wins, team2_wins = map(int, most_common_score.split(':'))
    total_games = team1_wins + team2_wins
    
    print(f"\n가장 많이 나온 시리즈 스코어 ({most_common_score})의 경기별 득점:")
    for game_num in range(total_games):
        game_scores = [score for res in results 
                      if f"{res['team1_wins']}:{res['team2_wins']}" == most_common_score 
                      for score in res['series_scores'] 
                      if score['game_num'] == game_num + 1]
        
        if game_scores:
            team1_avg = np.mean([score['team1_score'] for score in game_scores])
            team2_avg = np.mean([score['team2_score'] for score in game_scores])
            print(f"경기 {game_num + 1}: {team1} {team1_avg:.1f} - {team2} {team2_avg:.1f}")

# 결승전 결과 출력
print("\n=== 결승전 결과 ===")
team1, team2 = finals_teams

team1_wins = sum(1 for res in finals_simulation_results if res['team1_wins'] > res['team2_wins'])
team2_wins = sum(1 for res in finals_simulation_results if res['team2_wins'] > res['team1_wins'])
team1_win_rate = team1_wins / SIMULATION_CONFIG['n_simulations'] * 100
team2_win_rate = team2_wins / SIMULATION_CONFIG['n_simulations'] * 100

print(f"\n승률:")
print(f"{team1}: {team1_win_rate:.1f}%")
print(f"{team2}: {team2_win_rate:.1f}%")

# 평균 득점 계산
team1_avg_score = np.mean([score['team1_score'] for res in finals_simulation_results for score in res['series_scores']])
team2_avg_score = np.mean([score['team2_score'] for res in finals_simulation_results for score in res['series_scores']])
team1_std_score = np.std([score['team1_score'] for res in finals_simulation_results for score in res['series_scores']])
team2_std_score = np.std([score['team2_score'] for res in finals_simulation_results for score in res['series_scores']])

print(f"\n평균 득점 (표준편차):")
print(f"{team1}: {team1_avg_score:.1f} (±{team1_std_score:.1f})")
print(f"{team2}: {team2_avg_score:.1f} (±{team2_std_score:.1f})")

# 시리즈 스코어 분포
series_scores = Counter([f"{res['team1_wins']}:{res['team2_wins']}" for res in finals_simulation_results])
print("\n시리즈 스코어 분포:")
for score, count in series_scores.most_common():
    print(f"{score}: {count/SIMULATION_CONFIG['n_simulations']*100:.1f}%")

# 가장 많이 나온 시리즈 스코어의 경기별 득점
most_common_score = series_scores.most_common(1)[0][0]
team1_wins, team2_wins = map(int, most_common_score.split(':'))
total_games = team1_wins + team2_wins

print(f"\n가장 많이 나온 시리즈 스코어 ({most_common_score})의 경기별 득점:")
for game_num in range(total_games):
    game_scores = [score for res in finals_simulation_results 
                  if f"{res['team1_wins']}:{res['team2_wins']}" == most_common_score 
                  for score in res['series_scores'] 
                  if score['game_num'] == game_num + 1]
    
    if game_scores:
        team1_avg = np.mean([score['team1_score'] for score in game_scores])
        team2_avg = np.mean([score['team2_score'] for score in game_scores])
        print(f"경기 {game_num + 1}: {team1} {team1_avg:.1f} - {team2} {team2_avg:.1f}")

# 결과 저장
results_df = pd.DataFrame({
    'stage': ['east_semifinal', 'west_semifinal', 'finals'],
    'team1': [semifinal_teams['east'][0], semifinal_teams['west'][0], finals_teams[0]],
    'team2': [semifinal_teams['east'][1], semifinal_teams['west'][1], finals_teams[1]],
    'team1_win_rate': [team1_win_rate, team1_win_rate, team1_win_rate],
    'team2_win_rate': [team2_win_rate, team2_win_rate, team2_win_rate]
})

results_df.to_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/결승예측/nba_playoffs_simulation_results_1.csv", index=False, encoding="utf-8-sig")


[I 2025-05-27 15:00:22,063] A new study created in memory with name: no-name-8878f4f0-9c7f-4e6e-b293-e3e166bfbd90



데이터 로드 중...

Optuna를 이용한 XGBoost GPU 모델 하이퍼파라미터 탐색 중...


[I 2025-05-27 15:00:30,082] Trial 0 finished with value: 2.711875566467426 and parameters: {'n_estimators': 724, 'max_depth': 4, 'learning_rate': 0.2614214118245349, 'subsample': 0.7161539986772057, 'colsample_bytree': 0.6380278536611118, 'reg_alpha': 0.2539418100511779, 'reg_lambda': 2.663053156601742}. Best is trial 0 with value: 2.711875566467426.
[I 2025-05-27 15:00:36,842] Trial 1 finished with value: 2.957685067640452 and parameters: {'n_estimators': 556, 'max_depth': 5, 'learning_rate': 0.10701805382473323, 'subsample': 0.63222754046485, 'colsample_bytree': 0.9395079276946698, 'reg_alpha': 0.8065276643748448, 'reg_lambda': 2.1169071235994075}. Best is trial 0 with value: 2.711875566467426.
[I 2025-05-27 15:00:54,391] Trial 2 finished with value: 3.6137850299555554 and parameters: {'n_estimators': 566, 'max_depth': 9, 'learning_rate': 0.21527803140371432, 'subsample': 0.8150226808243266, 'colsample_bytree': 0.901910775313997, 'reg_alpha': 0.5321997287358436, 'reg_lambda': 0.94176


✅ Optuna가 선택한 최적 XGBoost GPU 모델 파라미터:
{'n_estimators': 787, 'max_depth': 3, 'learning_rate': 0.29887876990499646, 'subsample': 0.6568782860652974, 'colsample_bytree': 0.9857860737885639, 'reg_alpha': 0.4118810896084141, 'reg_lambda': 2.724415550604004}

✅ XGBoost GPU 모델 성능 평가

[Train Performance]
RMSE: 1.1797
MAE: 0.9126
R²: 0.9914

[Valid Performance]
RMSE: 1.7361
MAE: 1.3246
R²: 0.9817

4강 및 결승 진출팀 필터링 중...

4강 팀별 데이터 수:
TEAM_NAME
new york knicks           89
indiana pacers            88
minnesota timberwolves    88
oklahoma city thunder     88
Name: count, dtype: int64

4강 시뮬레이션 중...

EAST 컨퍼런스 4강 시뮬레이션 중...


east 시뮬레이션: 100%|██████████| 1000/1000 [03:35<00:00,  4.65it/s]



WEST 컨퍼런스 4강 시뮬레이션 중...


west 시뮬레이션: 100%|██████████| 1000/1000 [03:38<00:00,  4.58it/s]



결승 진출팀: oklahoma city thunder vs indiana pacers

결승전 시뮬레이션 중...


결승전 시뮬레이션: 100%|██████████| 1000/1000 [03:35<00:00,  4.64it/s]


=== NBA 4강 및 결승전 시뮬레이션 결과 ===

EAST 컨퍼런스 4강 결과:

승률:
indiana pacers: 76.9%
new york knicks: 23.1%

평균 득점 (표준편차):
indiana pacers: 123.1 (±14.8)
new york knicks: 117.2 (±13.7)

시리즈 스코어 분포:
4:1: 25.2%
4:2: 22.0%
4:0: 16.2%
4:3: 13.5%
3:4: 10.2%
2:4: 7.5%
1:4: 3.4%
0:4: 2.0%

가장 많이 나온 시리즈 스코어 (4:1)의 경기별 득점:
경기 1: indiana pacers 126.1 - new york knicks 114.6
경기 2: indiana pacers 126.3 - new york knicks 114.6
경기 3: indiana pacers 122.5 - new york knicks 114.8
경기 4: indiana pacers 122.8 - new york knicks 115.7
경기 5: indiana pacers 131.5 - new york knicks 113.3

WEST 컨퍼런스 4강 결과:

승률:
oklahoma city thunder: 69.4%
minnesota timberwolves: 30.6%

평균 득점 (표준편차):
oklahoma city thunder: 121.2 (±14.6)
minnesota timberwolves: 115.5 (±11.4)

시리즈 스코어 분포:
4:2: 22.6%
4:1: 19.4%
4:0: 14.2%
3:4: 14.2%
4:3: 13.2%
2:4: 7.6%
1:4: 6.0%
0:4: 2.8%

가장 많이 나온 시리즈 스코어 (4:2)의 경기별 득점:
경기 1: oklahoma city thunder 122.4 - minnesota timberwolves 116.1
경기 2: oklahoma city thunder 123.7 - minnesota timberwolves 114.7
경기 3: 

### RF

In [24]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
import optuna
from tqdm import tqdm
import warnings
from collections import Counter

warnings.filterwarnings('ignore', category=UserWarning)

# ------------------------
# [1] 데이터 로드 및 전처리
# ------------------------
print("\n데이터 로드 중...")
train = pd.read_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/상준's 데이터/2017_2023_정규_플옵_재구성완료.csv")
valid = pd.read_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/2024_25_파생변수_포함_WIN_PROB_정제최종.csv")

train["TEAM_NAME"] = train["TEAM_NAME"].str.lower()
valid["TEAM_NAME"] = valid["TEAM_NAME"].str.lower()

base_features = [
    'FGM', 'PLUS_MINUS_x', 'FG_PCT_x', 'AST_x', 'FG3M', 'DREB', 'REB_x'
]

def create_features(df):
    df['GAME_NUM'] = df.groupby(['SEASON', 'TEAM_NAME']).cumcount() + 1
    df['SEASON_PROGRESS'] = df['GAME_NUM'] / 82
    df['OPP_PTS_DIFF'] = df['PTS_x'] - df['OPP_PTS']
    df['OPP_PTS_ALLOWED_DIFF'] = df['OPP_PTS'] - df['PTS_x']
    df['FG_EFFICIENCY'] = df['FGM'] / df['FGA']
    df['FG3_EFFICIENCY'] = df['FG3M'] / df['FG3A']
    df['SCORING_PACE'] = df['PTS_x'] / df['GAME_NUM']
    df['DEFENSE_RATING'] = df['OPP_PTS'] / df['GAME_NUM']
    df['HOME_ADVANTAGE'] = df['HOME'].astype(int)
    df['RECENT_PTS_AVG'] = df.groupby('TEAM_NAME')['PTS_x'].rolling(5).mean().reset_index(0, drop=True)
    df['RECENT_OPP_PTS_AVG'] = df.groupby('TEAM_NAME')['OPP_PTS'].rolling(5).mean().reset_index(0, drop=True)
    df['RECENT_WIN_RATE'] = df.groupby('TEAM_NAME')['WIN'].rolling(5).mean().reset_index(0, drop=True)
    return df

train = create_features(train)
valid = create_features(valid)

features = base_features + [
    'SEASON_PROGRESS', 'OPP_PTS_DIFF', 'OPP_PTS_ALLOWED_DIFF',
    'HOME_ADVANTAGE', 'FG_EFFICIENCY', 'FG3_EFFICIENCY',
    'SCORING_PACE', 'DEFENSE_RATING', 'RECENT_PTS_AVG',
    'RECENT_OPP_PTS_AVG', 'RECENT_WIN_RATE'
]

train = train.dropna(subset=features + ['PTS_x']).reset_index(drop=True)
valid = valid.dropna(subset=features + ['PTS_x']).reset_index(drop=True)

scaler = RobustScaler()
train[features] = scaler.fit_transform(train[features])
valid[features] = scaler.transform(valid[features])

# ------------------------
# [2] Optuna로 RF 모델 튜닝
# ------------------------
def objective_rf(trial):
    model = RandomForestRegressor(
        n_estimators=trial.suggest_int('n_estimators', 300, 800),
        max_depth=trial.suggest_int('max_depth', 6, 20),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 10),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 5),
        random_state=42,
        n_jobs=-1
    )
    score = cross_val_score(model, train[features], train['PTS_x'], scoring='neg_root_mean_squared_error', cv=3)
    return -score.mean()

print("\nOptuna로 RandomForest 튜닝 중...")
study_rf = optuna.create_study(direction="minimize")
study_rf.optimize(objective_rf, n_trials=500)

print("\n✅ 최적 RF 하이퍼파라미터:")
print(study_rf.best_params)

best_model = RandomForestRegressor(
    **study_rf.best_params,
    random_state=42,
    n_jobs=-1
)
best_model.fit(train[features], train['PTS_x'])

print("\n✅ RF 모델 성능 평가")
y_pred_train = best_model.predict(train[features])
y_pred_valid = best_model.predict(valid[features])

print("\n[Train Performance]")
print(f"RMSE: {np.sqrt(mean_squared_error(train['PTS_x'], y_pred_train)):.4f}")
print(f"MAE: {mean_absolute_error(train['PTS_x'], y_pred_train):.4f}")
print(f"R²: {r2_score(train['PTS_x'], y_pred_train):.4f}")

print("\n[Valid Performance]")
print(f"RMSE: {np.sqrt(mean_squared_error(valid['PTS_x'], y_pred_valid)):.4f}")
print(f"MAE: {mean_absolute_error(valid['PTS_x'], y_pred_valid):.4f}")
print(f"R²: {r2_score(valid['PTS_x'], y_pred_valid):.4f}")

# ------------------------
# [3] 시뮬레이션 설정값 정의 (noise_std 증가)
# ------------------------
SIMULATION_CONFIG = {
    'n_simulations': 1000,
    'noise_std': 4.5,
    'home_advantage': {
        'base': 3.0,
        'playoff': 1.5
    },
    'recent_games_weight': 0.7,
    'min_games': 10
}


# 이후 시뮬레이션 함수 및 실행 코드는 기존 XGB 버전 구조에 그대로 연결해 사용할 수 있습니다.
# simulate_game()에서 best_model.predict(...) 부분만 그대로 RF 모델에 적용됩니다.
# ------------------------
# ------------------------
# [6] 4강 및 결승 진출팀 필터링
# ------------------------
print("\n4강 및 결승 진출팀 필터링 중...")

# 4강 팀 정의
semifinal_teams = {
    "east": ["indiana pacers", "new york knicks"],
    "west": ["oklahoma city thunder", "minnesota timberwolves"]
}

# 4강 데이터 필터링
valid_semifinals = valid[valid["TEAM_NAME"].isin([team for teams in semifinal_teams.values() for team in teams])].copy()

# 팀별 데이터 수 출력
print("\n4강 팀별 데이터 수:")
print(valid_semifinals["TEAM_NAME"].value_counts())

# 데이터 밸런싱 함수
def balance_team_data(df, team, min_games):
    team_data = df[df["TEAM_NAME"] == team]
    if len(team_data) < min_games:
        recent_data = team_data.sort_values('GAME_NUM', ascending=False).head(5)
        recent_avg = recent_data[features].mean()
        additional_data = pd.DataFrame([recent_avg] * (min_games - len(team_data)))
        additional_data["TEAM_NAME"] = team
        return pd.concat([df, additional_data], ignore_index=True)
    return df

# 4강 데이터 밸런싱
for conference in semifinal_teams:
    for team in semifinal_teams[conference]:
        valid_semifinals = balance_team_data(valid_semifinals, team, SIMULATION_CONFIG['min_games'])

# ------------------------
# [7] 4강 시뮬레이션
# ------------------------
print("\n4강 시뮬레이션 중...")

def add_noise_to_score(score, std=SIMULATION_CONFIG['noise_std']):
    return score + np.random.normal(0, std)

def get_home_advantage(game_num, is_playoff=True):
    base_advantage = SIMULATION_CONFIG['home_advantage']['base']
    if is_playoff:
        base_advantage += SIMULATION_CONFIG['home_advantage']['playoff']
    return base_advantage

def simulate_game(team1_data, team2_data, features, game_num, is_home=True):
    if len(team1_data) > 0 and len(team2_data) > 0:
        n1 = len(team1_data)
        n2 = len(team2_data)
        series_weight = 1.0 + (game_num * 0.1)
        weights1 = np.exp(-np.arange(n1) * (1 - SIMULATION_CONFIG['recent_games_weight'])) * series_weight
        weights1 = weights1 / weights1.sum()
        weights2 = np.exp(-np.arange(n2) * (1 - SIMULATION_CONFIG['recent_games_weight'])) * series_weight
        weights2 = weights2 / weights2.sum()
        t1_row = team1_data.sample(n=1, weights=weights1, random_state=np.random.randint(0, 10000))
        t2_row = team2_data.sample(n=1, weights=weights2, random_state=np.random.randint(0, 10000))
        home_advantage = get_home_advantage(game_num) if is_home else 0.0
        if game_num >= 4:
            home_advantage *= 1.2
        t1_score = add_noise_to_score(best_model.predict(t1_row[features])[0] + home_advantage)
        t2_score = add_noise_to_score(best_model.predict(t2_row[features])[0])

        t1_score = max(70, min(150, t1_score))
        t2_score = max(70, min(150, t2_score))
        return t1_score, t2_score
    return None, None

def run_semifinal_simulation(team1, team2, team1_data, team2_data, features):
    team1_wins = 0
    team2_wins = 0
    series_scores = []
    
    for game_num in range(7):
        if team1_wins >= 4 or team2_wins >= 4:
            break
            
        is_home = (game_num < 2) or (game_num >= 4 and game_num < 6)
        t1_score, t2_score = simulate_game(team1_data, team2_data, features, game_num, is_home)
        
        if t1_score is not None and t2_score is not None:
            if t1_score > t2_score:
                team1_wins += 1
                winner = team1
            else:
                team2_wins += 1
                winner = team2
            series_scores.append({
                "team1_score": t1_score,
                "team2_score": t2_score,
                "winner": winner,
                "is_home": is_home,
                "game_num": game_num + 1
            })
    
    return {
        "team1": team1,
        "team2": team2,
        "team1_wins": team1_wins,
        "team2_wins": team2_wins,
        "series_scores": series_scores
    }

# 4강 시뮬레이션 실행
semifinal_results = {}
conference_winners = {}

for conference, teams in semifinal_teams.items():
    print(f"\n{conference.upper()} 컨퍼런스 4강 시뮬레이션 중...")
    simulation_results = []
    for _ in tqdm(range(SIMULATION_CONFIG['n_simulations']), desc=f"{conference} 시뮬레이션"):
        result = run_semifinal_simulation(
            teams[0],
            teams[1],
            valid_semifinals[valid_semifinals["TEAM_NAME"] == teams[0]],
            valid_semifinals[valid_semifinals["TEAM_NAME"] == teams[1]],
            features
        )
        simulation_results.append(result)
    semifinal_results[conference] = simulation_results
    
    # 컨퍼런스 승자 결정
    team1_wins = sum(1 for res in simulation_results if res['team1_wins'] > res['team2_wins'])
    team2_wins = sum(1 for res in simulation_results if res['team2_wins'] > res['team1_wins'])
    conference_winners[conference] = teams[0] if team1_wins > team2_wins else teams[1]

# 결승 진출팀 결정
finals_teams = [conference_winners['west'], conference_winners['east']]
print(f"\n결승 진출팀: {finals_teams[0]} vs {finals_teams[1]}")

# 결승 데이터 필터링
valid_finals = valid[valid["TEAM_NAME"].isin(finals_teams)].copy()

# 결승 데이터 밸런싱
for team in finals_teams:
    valid_finals = balance_team_data(valid_finals, team, SIMULATION_CONFIG['min_games'])

# ------------------------
# [8] 결승전 시뮬레이션
# ------------------------
print("\n결승전 시뮬레이션 중...")
finals_simulation_results = []
for _ in tqdm(range(SIMULATION_CONFIG['n_simulations']), desc="결승전 시뮬레이션"):
    result = run_semifinal_simulation(
        finals_teams[0],
        finals_teams[1],
        valid_finals[valid_finals["TEAM_NAME"] == finals_teams[0]],
        valid_finals[valid_finals["TEAM_NAME"] == finals_teams[1]],
        features
    )
    finals_simulation_results.append(result)

# ------------------------
# [9] 결과 출력 및 저장
# ------------------------
print("\n=== NBA 4강 및 결승전 시뮬레이션 결과 ===")

# 4강 결과 출력
for conference, results in semifinal_results.items():
    print(f"\n{conference.upper()} 컨퍼런스 4강 결과:")
    team1, team2 = semifinal_teams[conference]
    
    team1_wins = sum(1 for res in results if res['team1_wins'] > res['team2_wins'])
    team2_wins = sum(1 for res in results if res['team2_wins'] > res['team1_wins'])
    team1_win_rate = team1_wins / SIMULATION_CONFIG['n_simulations'] * 100
    team2_win_rate = team2_wins / SIMULATION_CONFIG['n_simulations'] * 100
    
    print(f"\n승률:")
    print(f"{team1}: {team1_win_rate:.1f}%")
    print(f"{team2}: {team2_win_rate:.1f}%")
    
    # 평균 득점 계산
    team1_avg_score = np.mean([score['team1_score'] for res in results for score in res['series_scores']])
    team2_avg_score = np.mean([score['team2_score'] for res in results for score in res['series_scores']])
    team1_std_score = np.std([score['team1_score'] for res in results for score in res['series_scores']])
    team2_std_score = np.std([score['team2_score'] for res in results for score in res['series_scores']])
    
    print(f"\n평균 득점 (표준편차):")
    print(f"{team1}: {team1_avg_score:.1f} (±{team1_std_score:.1f})")
    print(f"{team2}: {team2_avg_score:.1f} (±{team2_std_score:.1f})")
    
    # 시리즈 스코어 분포
    series_scores = Counter([f"{res['team1_wins']}:{res['team2_wins']}" for res in results])
    print("\n시리즈 스코어 분포:")
    for score, count in series_scores.most_common():
        print(f"{score}: {count/SIMULATION_CONFIG['n_simulations']*100:.1f}%")
    
    # 가장 많이 나온 시리즈 스코어의 경기별 득점
    most_common_score = series_scores.most_common(1)[0][0]
    team1_wins, team2_wins = map(int, most_common_score.split(':'))
    total_games = team1_wins + team2_wins
    
    print(f"\n가장 많이 나온 시리즈 스코어 ({most_common_score})의 경기별 득점:")
    for game_num in range(total_games):
        game_scores = [score for res in results 
                      if f"{res['team1_wins']}:{res['team2_wins']}" == most_common_score 
                      for score in res['series_scores'] 
                      if score['game_num'] == game_num + 1]
        
        if game_scores:
            team1_avg = np.mean([score['team1_score'] for score in game_scores])
            team2_avg = np.mean([score['team2_score'] for score in game_scores])
            print(f"경기 {game_num + 1}: {team1} {team1_avg:.1f} - {team2} {team2_avg:.1f}")

# 결승전 결과 출력
print("\n=== 결승전 결과 ===")
team1, team2 = finals_teams

team1_wins = sum(1 for res in finals_simulation_results if res['team1_wins'] > res['team2_wins'])
team2_wins = sum(1 for res in finals_simulation_results if res['team2_wins'] > res['team1_wins'])
team1_win_rate = team1_wins / SIMULATION_CONFIG['n_simulations'] * 100
team2_win_rate = team2_wins / SIMULATION_CONFIG['n_simulations'] * 100

print(f"\n승률:")
print(f"{team1}: {team1_win_rate:.1f}%")
print(f"{team2}: {team2_win_rate:.1f}%")

# 평균 득점 계산
team1_avg_score = np.mean([score['team1_score'] for res in finals_simulation_results for score in res['series_scores']])
team2_avg_score = np.mean([score['team2_score'] for res in finals_simulation_results for score in res['series_scores']])
team1_std_score = np.std([score['team1_score'] for res in finals_simulation_results for score in res['series_scores']])
team2_std_score = np.std([score['team2_score'] for res in finals_simulation_results for score in res['series_scores']])

print(f"\n평균 득점 (표준편차):")
print(f"{team1}: {team1_avg_score:.1f} (±{team1_std_score:.1f})")
print(f"{team2}: {team2_avg_score:.1f} (±{team2_std_score:.1f})")

# 시리즈 스코어 분포
series_scores = Counter([f"{res['team1_wins']}:{res['team2_wins']}" for res in finals_simulation_results])
print("\n시리즈 스코어 분포:")
for score, count in series_scores.most_common():
    print(f"{score}: {count/SIMULATION_CONFIG['n_simulations']*100:.1f}%")

# 가장 많이 나온 시리즈 스코어의 경기별 득점
most_common_score = series_scores.most_common(1)[0][0]
team1_wins, team2_wins = map(int, most_common_score.split(':'))
total_games = team1_wins + team2_wins

print(f"\n가장 많이 나온 시리즈 스코어 ({most_common_score})의 경기별 득점:")
for game_num in range(total_games):
    game_scores = [score for res in finals_simulation_results 
                  if f"{res['team1_wins']}:{res['team2_wins']}" == most_common_score 
                  for score in res['series_scores'] 
                  if score['game_num'] == game_num + 1]
    
    if game_scores:
        team1_avg = np.mean([score['team1_score'] for score in game_scores])
        team2_avg = np.mean([score['team2_score'] for score in game_scores])
        print(f"경기 {game_num + 1}: {team1} {team1_avg:.1f} - {team2} {team2_avg:.1f}")

# 결과 저장
results_df = pd.DataFrame({
    'stage': ['east_semifinal', 'west_semifinal', 'finals'],
    'team1': [semifinal_teams['east'][0], semifinal_teams['west'][0], finals_teams[0]],
    'team2': [semifinal_teams['east'][1], semifinal_teams['west'][1], finals_teams[1]],
    'team1_win_rate': [team1_win_rate, team1_win_rate, team1_win_rate],
    'team2_win_rate': [team2_win_rate, team2_win_rate, team2_win_rate]
})

results_df.to_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/결승예측/nba_playoffs_simulation_results_2.csv", index=False, encoding="utf-8-sig")



데이터 로드 중...


[I 2025-05-27 16:14:52,551] A new study created in memory with name: no-name-feca97f2-7209-4dd9-9471-87bc2e8c6d48



Optuna로 RandomForest 튜닝 중...


[I 2025-05-27 16:14:58,738] Trial 0 finished with value: 5.466515342384099 and parameters: {'n_estimators': 315, 'max_depth': 7, 'min_samples_split': 10, 'min_samples_leaf': 4}. Best is trial 0 with value: 5.466515342384099.
[I 2025-05-27 16:15:36,626] Trial 1 finished with value: 5.270773709600574 and parameters: {'n_estimators': 714, 'max_depth': 18, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 1 with value: 5.270773709600574.
[I 2025-05-27 16:15:55,277] Trial 2 finished with value: 5.278785622474062 and parameters: {'n_estimators': 404, 'max_depth': 14, 'min_samples_split': 6, 'min_samples_leaf': 3}. Best is trial 1 with value: 5.270773709600574.
[I 2025-05-27 16:16:03,631] Trial 3 finished with value: 5.466813678329665 and parameters: {'n_estimators': 439, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 5}. Best is trial 1 with value: 5.270773709600574.
[I 2025-05-27 16:16:27,754] Trial 4 finished with value: 5.314895408220273 and parameters: {'n_estima


✅ 최적 RF 하이퍼파라미터:
{'n_estimators': 683, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 3}

✅ RF 모델 성능 평가

[Train Performance]
RMSE: 2.8150
MAE: 2.1696
R²: 0.9510

[Valid Performance]
RMSE: 5.0634
MAE: 4.0670
R²: 0.8444

4강 및 결승 진출팀 필터링 중...

4강 팀별 데이터 수:
TEAM_NAME
new york knicks           89
indiana pacers            88
minnesota timberwolves    88
oklahoma city thunder     88
Name: count, dtype: int64

4강 시뮬레이션 중...

EAST 컨퍼런스 4강 시뮬레이션 중...


...


WEST 컨퍼런스 4강 시뮬레이션 중...


west 시뮬레이션: 100%|██████████| 1000/1000 [10:04<00:00,  1.65it/s]



결승 진출팀: oklahoma city thunder vs indiana pacers

결승전 시뮬레이션 중...


결승전 시뮬레이션: 100%|██████████| 1000/1000 [10:06<00:00,  1.65it/s]


=== NBA 4강 및 결승전 시뮬레이션 결과 ===

EAST 컨퍼런스 4강 결과:

승률:
indiana pacers: 81.0%
new york knicks: 19.0%

평균 득점 (표준편차):
indiana pacers: 125.4 (±12.6)
new york knicks: 118.4 (±13.1)

시리즈 스코어 분포:
4:1: 24.4%
4:2: 23.7%
4:0: 18.8%
4:3: 14.1%
3:4: 8.6%
2:4: 5.8%
1:4: 2.9%
0:4: 1.7%

가장 많이 나온 시리즈 스코어 (4:1)의 경기별 득점:
경기 1: indiana pacers 128.1 - new york knicks 116.6
경기 2: indiana pacers 128.2 - new york knicks 116.2
경기 3: indiana pacers 122.7 - new york knicks 116.4
경기 4: indiana pacers 124.3 - new york knicks 118.0
경기 5: indiana pacers 131.5 - new york knicks 114.0

WEST 컨퍼런스 4강 결과:

승률:
oklahoma city thunder: 70.5%
minnesota timberwolves: 29.5%

평균 득점 (표준편차):
oklahoma city thunder: 121.7 (±12.3)
minnesota timberwolves: 116.6 (±11.0)

시리즈 스코어 분포:
4:1: 24.2%
4:2: 21.1%
4:3: 14.4%
3:4: 13.6%
4:0: 10.8%
2:4: 8.2%
1:4: 5.5%
0:4: 2.2%

가장 많이 나온 시리즈 스코어 (4:1)의 경기별 득점:
경기 1: oklahoma city thunder 126.1 - minnesota timberwolves 113.0
경기 2: oklahoma city thunder 126.6 - minnesota timberwolves 115.7
경기 3: o

### LGBM

In [27]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
import optuna
from tqdm import tqdm
import warnings
from collections import Counter

warnings.filterwarnings('ignore', category=UserWarning)

# ------------------------
# [1] 데이터 로드 및 전처리
# ------------------------
print("\n데이터 로드 중...")
train = pd.read_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/상준's 데이터/2017_2023_정규_플옵_재구성완료.csv")
valid = pd.read_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/2024_25_파생변수_포함_WIN_PROB_정제최종.csv")

train["TEAM_NAME"] = train["TEAM_NAME"].str.lower()
valid["TEAM_NAME"] = valid["TEAM_NAME"].str.lower()

base_features = [
    'FGM', 'PLUS_MINUS_x', 'FG_PCT_x', 'AST_x', 'FG3M', 'DREB', 'REB_x'
]

def create_features(df):
    df['GAME_NUM'] = df.groupby(['SEASON', 'TEAM_NAME']).cumcount() + 1
    df['SEASON_PROGRESS'] = df['GAME_NUM'] / 82
    df['OPP_PTS_DIFF'] = df['PTS_x'] - df['OPP_PTS']
    df['OPP_PTS_ALLOWED_DIFF'] = df['OPP_PTS'] - df['PTS_x']
    df['FG_EFFICIENCY'] = df['FGM'] / df['FGA']
    df['FG3_EFFICIENCY'] = df['FG3M'] / df['FG3A']
    df['SCORING_PACE'] = df['PTS_x'] / df['GAME_NUM']
    df['DEFENSE_RATING'] = df['OPP_PTS'] / df['GAME_NUM']
    df['HOME_ADVANTAGE'] = df['HOME'].astype(int)
    df['RECENT_PTS_AVG'] = df.groupby('TEAM_NAME')['PTS_x'].rolling(5).mean().reset_index(0, drop=True)
    df['RECENT_OPP_PTS_AVG'] = df.groupby('TEAM_NAME')['OPP_PTS'].rolling(5).mean().reset_index(0, drop=True)
    df['RECENT_WIN_RATE'] = df.groupby('TEAM_NAME')['WIN'].rolling(5).mean().reset_index(0, drop=True)
    return df

train = create_features(train)
valid = create_features(valid)

features = base_features + [
    'SEASON_PROGRESS', 'OPP_PTS_DIFF', 'OPP_PTS_ALLOWED_DIFF',
    'HOME_ADVANTAGE', 'FG_EFFICIENCY', 'FG3_EFFICIENCY',
    'SCORING_PACE', 'DEFENSE_RATING', 'RECENT_PTS_AVG',
    'RECENT_OPP_PTS_AVG', 'RECENT_WIN_RATE'
]

train = train.dropna(subset=features + ['PTS_x']).reset_index(drop=True)
valid = valid.dropna(subset=features + ['PTS_x']).reset_index(drop=True)

scaler = RobustScaler()
train[features] = scaler.fit_transform(train[features])
valid[features] = scaler.transform(valid[features])

# ------------------------
# [2] Optuna + LGBM 모델 학습 및 검증
# ------------------------
def objective_lgbm(trial):
    model = LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 300, 800),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 0.0, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.0, 3.0),
        device="gpu",
        random_state=42,
        verbosity=-1
    )
    score = cross_val_score(model, train[features], train['PTS_x'], scoring="neg_root_mean_squared_error", cv=5)
    return -score.mean()

print("\nOptuna를 이용한 LGBM GPU 모델 하이퍼파라미터 탐색 중...")
study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=500)

print("\n✅ Optuna가 선택한 최적 LGBM GPU 모델 파라미터:")
print(study_lgbm.best_params)

best_model = LGBMRegressor(
    **study_lgbm.best_params,
    device="gpu",
    random_state=42,
    verbosity=-1
)
best_model.fit(
    train[features], train['PTS_x'],
    eval_set=[(valid[features], valid['PTS_x'])],
    verbose=False
)

print("\n✅ LGBM GPU 모델 성능 평가")
y_pred_train = best_model.predict(train[features])
y_pred_valid = best_model.predict(valid[features])

print("\n[Train Performance]")
print(f"RMSE: {np.sqrt(mean_squared_error(train['PTS_x'], y_pred_train)):.4f}")
print(f"MAE: {mean_absolute_error(train['PTS_x'], y_pred_train):.4f}")
print(f"R²: {r2_score(train['PTS_x'], y_pred_train):.4f}")

print("\n[Valid Performance]")
print(f"RMSE: {np.sqrt(mean_squared_error(valid['PTS_x'], y_pred_valid)):.4f}")
print(f"MAE: {mean_absolute_error(valid['PTS_x'], y_pred_valid):.4f}")
print(f"R²: {r2_score(valid['PTS_x'], y_pred_valid):.4f}")

# ------------------------
# [3] 시뮬레이션 설정값 정의 (noise_std 증가)
# ------------------------
SIMULATION_CONFIG = {
    'n_simulations': 1000,
    'noise_std': 4.5,
    'home_advantage': {
        'base': 3.0,
        'playoff': 1.5
    },
    'recent_games_weight': 0.7,
    'min_games': 10
}
# ------------------------
# [6] 4강 및 결승 진출팀 필터링
# ------------------------
print("\n4강 및 결승 진출팀 필터링 중...")

# 4강 팀 정의
semifinal_teams = {
    "east": ["indiana pacers", "new york knicks"],
    "west": ["oklahoma city thunder", "minnesota timberwolves"]
}

# 4강 데이터 필터링
valid_semifinals = valid[valid["TEAM_NAME"].isin([team for teams in semifinal_teams.values() for team in teams])].copy()

# 팀별 데이터 수 출력
print("\n4강 팀별 데이터 수:")
print(valid_semifinals["TEAM_NAME"].value_counts())

# 데이터 밸런싱 함수
def balance_team_data(df, team, min_games):
    team_data = df[df["TEAM_NAME"] == team]
    if len(team_data) < min_games:
        recent_data = team_data.sort_values('GAME_NUM', ascending=False).head(5)
        recent_avg = recent_data[features].mean()
        additional_data = pd.DataFrame([recent_avg] * (min_games - len(team_data)))
        additional_data["TEAM_NAME"] = team
        return pd.concat([df, additional_data], ignore_index=True)
    return df

# 4강 데이터 밸런싱
for conference in semifinal_teams:
    for team in semifinal_teams[conference]:
        valid_semifinals = balance_team_data(valid_semifinals, team, SIMULATION_CONFIG['min_games'])

# ------------------------
# [7] 4강 시뮬레이션
# ------------------------
print("\n4강 시뮬레이션 중...")

def add_noise_to_score(score, std=SIMULATION_CONFIG['noise_std']):
    return score + np.random.normal(0, std)

def get_home_advantage(game_num, is_playoff=True):
    base_advantage = SIMULATION_CONFIG['home_advantage']['base']
    if is_playoff:
        base_advantage += SIMULATION_CONFIG['home_advantage']['playoff']
    return base_advantage

def simulate_game(team1_data, team2_data, features, game_num, is_home=True):
    if len(team1_data) > 0 and len(team2_data) > 0:
        n1 = len(team1_data)
        n2 = len(team2_data)
        series_weight = 1.0 + (game_num * 0.1)
        weights1 = np.exp(-np.arange(n1) * (1 - SIMULATION_CONFIG['recent_games_weight'])) * series_weight
        weights1 = weights1 / weights1.sum()
        weights2 = np.exp(-np.arange(n2) * (1 - SIMULATION_CONFIG['recent_games_weight'])) * series_weight
        weights2 = weights2 / weights2.sum()
        t1_row = team1_data.sample(n=1, weights=weights1, random_state=np.random.randint(0, 10000))
        t2_row = team2_data.sample(n=1, weights=weights2, random_state=np.random.randint(0, 10000))
        home_advantage = get_home_advantage(game_num) if is_home else 0.0
        if game_num >= 4:
            home_advantage *= 1.2
        t1_score = add_noise_to_score(best_model.predict(t1_row[features])[0] + home_advantage)
        t2_score = add_noise_to_score(best_model.predict(t2_row[features])[0])

        t1_score = max(70, min(150, t1_score))
        t2_score = max(70, min(150, t2_score))
        return t1_score, t2_score
    return None, None

def run_semifinal_simulation(team1, team2, team1_data, team2_data, features):
    team1_wins = 0
    team2_wins = 0
    series_scores = []
    
    for game_num in range(7):
        if team1_wins >= 4 or team2_wins >= 4:
            break
            
        is_home = (game_num < 2) or (game_num >= 4 and game_num < 6)
        t1_score, t2_score = simulate_game(team1_data, team2_data, features, game_num, is_home)
        
        if t1_score is not None and t2_score is not None:
            if t1_score > t2_score:
                team1_wins += 1
                winner = team1
            else:
                team2_wins += 1
                winner = team2
            series_scores.append({
                "team1_score": t1_score,
                "team2_score": t2_score,
                "winner": winner,
                "is_home": is_home,
                "game_num": game_num + 1
            })
    
    return {
        "team1": team1,
        "team2": team2,
        "team1_wins": team1_wins,
        "team2_wins": team2_wins,
        "series_scores": series_scores
    }

# 4강 시뮬레이션 실행
semifinal_results = {}
conference_winners = {}

for conference, teams in semifinal_teams.items():
    print(f"\n{conference.upper()} 컨퍼런스 4강 시뮬레이션 중...")
    simulation_results = []
    for _ in tqdm(range(SIMULATION_CONFIG['n_simulations']), desc=f"{conference} 시뮬레이션"):
        result = run_semifinal_simulation(
            teams[0],
            teams[1],
            valid_semifinals[valid_semifinals["TEAM_NAME"] == teams[0]],
            valid_semifinals[valid_semifinals["TEAM_NAME"] == teams[1]],
            features
        )
        simulation_results.append(result)
    semifinal_results[conference] = simulation_results
    
    # 컨퍼런스 승자 결정
    team1_wins = sum(1 for res in simulation_results if res['team1_wins'] > res['team2_wins'])
    team2_wins = sum(1 for res in simulation_results if res['team2_wins'] > res['team1_wins'])
    conference_winners[conference] = teams[0] if team1_wins > team2_wins else teams[1]

# 결승 진출팀 결정
finals_teams = [conference_winners['west'], conference_winners['east']]
print(f"\n결승 진출팀: {finals_teams[0]} vs {finals_teams[1]}")

# 결승 데이터 필터링
valid_finals = valid[valid["TEAM_NAME"].isin(finals_teams)].copy()

# 결승 데이터 밸런싱
for team in finals_teams:
    valid_finals = balance_team_data(valid_finals, team, SIMULATION_CONFIG['min_games'])

# ------------------------
# [8] 결승전 시뮬레이션
# ------------------------
print("\n결승전 시뮬레이션 중...")
finals_simulation_results = []
for _ in tqdm(range(SIMULATION_CONFIG['n_simulations']), desc="결승전 시뮬레이션"):
    result = run_semifinal_simulation(
        finals_teams[0],
        finals_teams[1],
        valid_finals[valid_finals["TEAM_NAME"] == finals_teams[0]],
        valid_finals[valid_finals["TEAM_NAME"] == finals_teams[1]],
        features
    )
    finals_simulation_results.append(result)

# ------------------------
# [9] 결과 출력 및 저장
# ------------------------
print("\n=== NBA 4강 및 결승전 시뮬레이션 결과 ===")

# 4강 결과 출력
for conference, results in semifinal_results.items():
    print(f"\n{conference.upper()} 컨퍼런스 4강 결과:")
    team1, team2 = semifinal_teams[conference]
    
    team1_wins = sum(1 for res in results if res['team1_wins'] > res['team2_wins'])
    team2_wins = sum(1 for res in results if res['team2_wins'] > res['team1_wins'])
    team1_win_rate = team1_wins / SIMULATION_CONFIG['n_simulations'] * 100
    team2_win_rate = team2_wins / SIMULATION_CONFIG['n_simulations'] * 100
    
    print(f"\n승률:")
    print(f"{team1}: {team1_win_rate:.1f}%")
    print(f"{team2}: {team2_win_rate:.1f}%")
    
    # 평균 득점 계산
    team1_avg_score = np.mean([score['team1_score'] for res in results for score in res['series_scores']])
    team2_avg_score = np.mean([score['team2_score'] for res in results for score in res['series_scores']])
    team1_std_score = np.std([score['team1_score'] for res in results for score in res['series_scores']])
    team2_std_score = np.std([score['team2_score'] for res in results for score in res['series_scores']])
    
    print(f"\n평균 득점 (표준편차):")
    print(f"{team1}: {team1_avg_score:.1f} (±{team1_std_score:.1f})")
    print(f"{team2}: {team2_avg_score:.1f} (±{team2_std_score:.1f})")
    
    # 시리즈 스코어 분포
    series_scores = Counter([f"{res['team1_wins']}:{res['team2_wins']}" for res in results])
    print("\n시리즈 스코어 분포:")
    for score, count in series_scores.most_common():
        print(f"{score}: {count/SIMULATION_CONFIG['n_simulations']*100:.1f}%")
    
    # 가장 많이 나온 시리즈 스코어의 경기별 득점
    most_common_score = series_scores.most_common(1)[0][0]
    team1_wins, team2_wins = map(int, most_common_score.split(':'))
    total_games = team1_wins + team2_wins
    
    print(f"\n가장 많이 나온 시리즈 스코어 ({most_common_score})의 경기별 득점:")
    for game_num in range(total_games):
        game_scores = [score for res in results 
                      if f"{res['team1_wins']}:{res['team2_wins']}" == most_common_score 
                      for score in res['series_scores'] 
                      if score['game_num'] == game_num + 1]
        
        if game_scores:
            team1_avg = np.mean([score['team1_score'] for score in game_scores])
            team2_avg = np.mean([score['team2_score'] for score in game_scores])
            print(f"경기 {game_num + 1}: {team1} {team1_avg:.1f} - {team2} {team2_avg:.1f}")

# 결승전 결과 출력
print("\n=== 결승전 결과 ===")
team1, team2 = finals_teams

team1_wins = sum(1 for res in finals_simulation_results if res['team1_wins'] > res['team2_wins'])
team2_wins = sum(1 for res in finals_simulation_results if res['team2_wins'] > res['team1_wins'])
team1_win_rate = team1_wins / SIMULATION_CONFIG['n_simulations'] * 100
team2_win_rate = team2_wins / SIMULATION_CONFIG['n_simulations'] * 100

print(f"\n승률:")
print(f"{team1}: {team1_win_rate:.1f}%")
print(f"{team2}: {team2_win_rate:.1f}%")

# 평균 득점 계산
team1_avg_score = np.mean([score['team1_score'] for res in finals_simulation_results for score in res['series_scores']])
team2_avg_score = np.mean([score['team2_score'] for res in finals_simulation_results for score in res['series_scores']])
team1_std_score = np.std([score['team1_score'] for res in finals_simulation_results for score in res['series_scores']])
team2_std_score = np.std([score['team2_score'] for res in finals_simulation_results for score in res['series_scores']])

print(f"\n평균 득점 (표준편차):")
print(f"{team1}: {team1_avg_score:.1f} (±{team1_std_score:.1f})")
print(f"{team2}: {team2_avg_score:.1f} (±{team2_std_score:.1f})")

# 시리즈 스코어 분포
series_scores = Counter([f"{res['team1_wins']}:{res['team2_wins']}" for res in finals_simulation_results])
print("\n시리즈 스코어 분포:")
for score, count in series_scores.most_common():
    print(f"{score}: {count/SIMULATION_CONFIG['n_simulations']*100:.1f}%")

# 가장 많이 나온 시리즈 스코어의 경기별 득점
most_common_score = series_scores.most_common(1)[0][0]
team1_wins, team2_wins = map(int, most_common_score.split(':'))
total_games = team1_wins + team2_wins

print(f"\n가장 많이 나온 시리즈 스코어 ({most_common_score})의 경기별 득점:")
for game_num in range(total_games):
    game_scores = [score for res in finals_simulation_results 
                  if f"{res['team1_wins']}:{res['team2_wins']}" == most_common_score 
                  for score in res['series_scores'] 
                  if score['game_num'] == game_num + 1]
    
    if game_scores:
        team1_avg = np.mean([score['team1_score'] for score in game_scores])
        team2_avg = np.mean([score['team2_score'] for score in game_scores])
        print(f"경기 {game_num + 1}: {team1} {team1_avg:.1f} - {team2} {team2_avg:.1f}")

# 결과 저장
results_df = pd.DataFrame({
    'stage': ['east_semifinal', 'west_semifinal', 'finals'],
    'team1': [semifinal_teams['east'][0], semifinal_teams['west'][0], finals_teams[0]],
    'team2': [semifinal_teams['east'][1], semifinal_teams['west'][1], finals_teams[1]],
    'team1_win_rate': [team1_win_rate, team1_win_rate, team1_win_rate],
    'team2_win_rate': [team2_win_rate, team2_win_rate, team2_win_rate]
})

results_df.to_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/결승예측/nba_playoffs_simulation_results_3.csv", index=False, encoding="utf-8-sig")

# 이후 시뮬레이션 함수 및 실행 코드는 기존 XGB/RF 구조와 동일하게 연결하여 사용 가능합니다.

[I 2025-05-27 23:02:24,138] A new study created in memory with name: no-name-b9c37577-5ef3-4668-8ff2-0ef213c60e13



데이터 로드 중...

Optuna를 이용한 LGBM GPU 모델 하이퍼파라미터 탐색 중...


[I 2025-05-27 23:02:33,906] Trial 0 finished with value: 4.926492731554777 and parameters: {'n_estimators': 534, 'max_depth': 5, 'learning_rate': 0.010077900057011811, 'subsample': 0.6103043685251834, 'colsample_bytree': 0.831380627294592, 'reg_alpha': 0.6724099192422627, 'reg_lambda': 0.7330376816180807}. Best is trial 0 with value: 4.926492731554777.
[I 2025-05-27 23:02:47,353] Trial 1 finished with value: 2.8730619072938794 and parameters: {'n_estimators': 611, 'max_depth': 9, 'learning_rate': 0.2202099284018442, 'subsample': 0.7060742440141754, 'colsample_bytree': 0.6703024232738412, 'reg_alpha': 0.21918007574719744, 'reg_lambda': 2.5386923903960428}. Best is trial 1 with value: 2.8730619072938794.
[I 2025-05-27 23:02:51,332] Trial 2 finished with value: 4.634794438139341 and parameters: {'n_estimators': 705, 'max_depth': 3, 'learning_rate': 0.026420125821589827, 'subsample': 0.8076015804263283, 'colsample_bytree': 0.8442687819262524, 'reg_alpha': 0.17689136334703603, 'reg_lambda':

KeyboardInterrupt: 

### CATBoost

In [26]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
import optuna
from tqdm import tqdm
import warnings
from collections import Counter

warnings.filterwarnings('ignore', category=UserWarning)

# ------------------------
# [1] 데이터 로드 및 전처리
# ------------------------
print("\n데이터 로드 중...")
train = pd.read_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/상준's 데이터/2017_2023_정규_플옵_재구성완료.csv")
valid = pd.read_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/2024_25_파생변수_포함_WIN_PROB_정제최종.csv")

train["TEAM_NAME"] = train["TEAM_NAME"].str.lower()
valid["TEAM_NAME"] = valid["TEAM_NAME"].str.lower()

base_features = [
    'FGM', 'PLUS_MINUS_x', 'FG_PCT_x', 'AST_x', 'FG3M', 'DREB', 'REB_x'
]

def create_features(df):
    df['GAME_NUM'] = df.groupby(['SEASON', 'TEAM_NAME']).cumcount() + 1
    df['SEASON_PROGRESS'] = df['GAME_NUM'] / 82
    df['OPP_PTS_DIFF'] = df['PTS_x'] - df['OPP_PTS']
    df['OPP_PTS_ALLOWED_DIFF'] = df['OPP_PTS'] - df['PTS_x']
    df['FG_EFFICIENCY'] = df['FGM'] / df['FGA']
    df['FG3_EFFICIENCY'] = df['FG3M'] / df['FG3A']
    df['SCORING_PACE'] = df['PTS_x'] / df['GAME_NUM']
    df['DEFENSE_RATING'] = df['OPP_PTS'] / df['GAME_NUM']
    df['HOME_ADVANTAGE'] = df['HOME'].astype(int)
    df['RECENT_PTS_AVG'] = df.groupby('TEAM_NAME')['PTS_x'].rolling(5).mean().reset_index(0, drop=True)
    df['RECENT_OPP_PTS_AVG'] = df.groupby('TEAM_NAME')['OPP_PTS'].rolling(5).mean().reset_index(0, drop=True)
    df['RECENT_WIN_RATE'] = df.groupby('TEAM_NAME')['WIN'].rolling(5).mean().reset_index(0, drop=True)
    return df

train = create_features(train)
valid = create_features(valid)

features = base_features + [
    'SEASON_PROGRESS', 'OPP_PTS_DIFF', 'OPP_PTS_ALLOWED_DIFF',
    'HOME_ADVANTAGE', 'FG_EFFICIENCY', 'FG3_EFFICIENCY',
    'SCORING_PACE', 'DEFENSE_RATING', 'RECENT_PTS_AVG',
    'RECENT_OPP_PTS_AVG', 'RECENT_WIN_RATE'
]

train = train.dropna(subset=features + ['PTS_x']).reset_index(drop=True)
valid = valid.dropna(subset=features + ['PTS_x']).reset_index(drop=True)

scaler = RobustScaler()
train[features] = scaler.fit_transform(train[features])
valid[features] = scaler.transform(valid[features])

# ------------------------
# [2] Optuna로 CatBoost 튜닝
# ------------------------
def objective_cat(trial):
    model = CatBoostRegressor(
        iterations=trial.suggest_int('iterations', 300, 800),
        depth=trial.suggest_int('depth', 4, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1.0, 5.0),
        loss_function='RMSE',
        task_type='GPU',
        verbose=0,
        random_seed=42
    )
    score = cross_val_score(model, train[features], train['PTS_x'], scoring='neg_root_mean_squared_error', cv=3)
    return -score.mean()

print("\nOptuna로 CatBoost 튜닝 중...")
study_cat = optuna.create_study(direction="minimize")
study_cat.optimize(objective_cat, n_trials=500)

print("\n✅ 최적 CatBoost 하이퍼파라미터:")
print(study_cat.best_params)

best_model = CatBoostRegressor(
    **study_cat.best_params,
    loss_function='RMSE',
    task_type='GPU',
    verbose=0,
    random_seed=42
)
best_model.fit(train[features], train['PTS_x'])

print("\n✅ CatBoost 모델 성능 평가")
y_pred_train = best_model.predict(train[features])
y_pred_valid = best_model.predict(valid[features])

print("\n[Train Performance]")
print(f"RMSE: {np.sqrt(mean_squared_error(train['PTS_x'], y_pred_train)):.4f}")
print(f"MAE: {mean_absolute_error(train['PTS_x'], y_pred_train):.4f}")
print(f"R²: {r2_score(train['PTS_x'], y_pred_train):.4f}")

print("\n[Valid Performance]")
print(f"RMSE: {np.sqrt(mean_squared_error(valid['PTS_x'], y_pred_valid)):.4f}")
print(f"MAE: {mean_absolute_error(valid['PTS_x'], y_pred_valid):.4f}")
print(f"R²: {r2_score(valid['PTS_x'], y_pred_valid):.4f}")

# ------------------------
# [3] 시뮬레이션 설정값 정의
# ------------------------
SIMULATION_CONFIG = {
    'n_simulations': 1000,
    'noise_std': 4.5,
    'home_advantage': {
        'base': 3.0,
        'playoff': 1.5
    },
    'recent_games_weight': 0.7,
    'min_games': 10
}
# 이후 시뮬레이션 함수 및 실행 코드는 기존 XGB 버전 구조에 그대로 연결해 사용할 수 있습니다.
# simulate_game()에서 best_model.predict(...) 부분만 그대로 CatBoost 모델에 적용됩니다.
# ------------------------
# [6] 4강 및 결승 진출팀 필터링
# ------------------------
print("\n4강 및 결승 진출팀 필터링 중...")

# 4강 팀 정의
semifinal_teams = {
    "east": ["indiana pacers", "new york knicks"],
    "west": ["oklahoma city thunder", "minnesota timberwolves"]
}

# 4강 데이터 필터링
valid_semifinals = valid[valid["TEAM_NAME"].isin([team for teams in semifinal_teams.values() for team in teams])].copy()

# 팀별 데이터 수 출력
print("\n4강 팀별 데이터 수:")
print(valid_semifinals["TEAM_NAME"].value_counts())

# 데이터 밸런싱 함수
def balance_team_data(df, team, min_games):
    team_data = df[df["TEAM_NAME"] == team]
    if len(team_data) < min_games:
        recent_data = team_data.sort_values('GAME_NUM', ascending=False).head(5)
        recent_avg = recent_data[features].mean()
        additional_data = pd.DataFrame([recent_avg] * (min_games - len(team_data)))
        additional_data["TEAM_NAME"] = team
        return pd.concat([df, additional_data], ignore_index=True)
    return df

# 4강 데이터 밸런싱
for conference in semifinal_teams:
    for team in semifinal_teams[conference]:
        valid_semifinals = balance_team_data(valid_semifinals, team, SIMULATION_CONFIG['min_games'])

# ------------------------
# [7] 4강 시뮬레이션
# ------------------------
print("\n4강 시뮬레이션 중...")

def add_noise_to_score(score, std=SIMULATION_CONFIG['noise_std']):
    return score + np.random.normal(0, std)

def get_home_advantage(game_num, is_playoff=True):
    base_advantage = SIMULATION_CONFIG['home_advantage']['base']
    if is_playoff:
        base_advantage += SIMULATION_CONFIG['home_advantage']['playoff']
    return base_advantage

def simulate_game(team1_data, team2_data, features, game_num, is_home=True):
    if len(team1_data) > 0 and len(team2_data) > 0:
        n1 = len(team1_data)
        n2 = len(team2_data)
        series_weight = 1.0 + (game_num * 0.1)
        weights1 = np.exp(-np.arange(n1) * (1 - SIMULATION_CONFIG['recent_games_weight'])) * series_weight
        weights1 = weights1 / weights1.sum()
        weights2 = np.exp(-np.arange(n2) * (1 - SIMULATION_CONFIG['recent_games_weight'])) * series_weight
        weights2 = weights2 / weights2.sum()
        t1_row = team1_data.sample(n=1, weights=weights1, random_state=np.random.randint(0, 10000))
        t2_row = team2_data.sample(n=1, weights=weights2, random_state=np.random.randint(0, 10000))
        home_advantage = get_home_advantage(game_num) if is_home else 0.0
        if game_num >= 4:
            home_advantage *= 1.2
        t1_score = add_noise_to_score(best_model.predict(t1_row[features])[0] + home_advantage)
        t2_score = add_noise_to_score(best_model.predict(t2_row[features])[0])

        t1_score = max(70, min(150, t1_score))
        t2_score = max(70, min(150, t2_score))
        return t1_score, t2_score
    return None, None

def run_semifinal_simulation(team1, team2, team1_data, team2_data, features):
    team1_wins = 0
    team2_wins = 0
    series_scores = []
    
    for game_num in range(7):
        if team1_wins >= 4 or team2_wins >= 4:
            break
            
        is_home = (game_num < 2) or (game_num >= 4 and game_num < 6)
        t1_score, t2_score = simulate_game(team1_data, team2_data, features, game_num, is_home)
        
        if t1_score is not None and t2_score is not None:
            if t1_score > t2_score:
                team1_wins += 1
                winner = team1
            else:
                team2_wins += 1
                winner = team2
            series_scores.append({
                "team1_score": t1_score,
                "team2_score": t2_score,
                "winner": winner,
                "is_home": is_home,
                "game_num": game_num + 1
            })
    
    return {
        "team1": team1,
        "team2": team2,
        "team1_wins": team1_wins,
        "team2_wins": team2_wins,
        "series_scores": series_scores
    }

# 4강 시뮬레이션 실행
semifinal_results = {}
conference_winners = {}

for conference, teams in semifinal_teams.items():
    print(f"\n{conference.upper()} 컨퍼런스 4강 시뮬레이션 중...")
    simulation_results = []
    for _ in tqdm(range(SIMULATION_CONFIG['n_simulations']), desc=f"{conference} 시뮬레이션"):
        result = run_semifinal_simulation(
            teams[0],
            teams[1],
            valid_semifinals[valid_semifinals["TEAM_NAME"] == teams[0]],
            valid_semifinals[valid_semifinals["TEAM_NAME"] == teams[1]],
            features
        )
        simulation_results.append(result)
    semifinal_results[conference] = simulation_results
    
    # 컨퍼런스 승자 결정
    team1_wins = sum(1 for res in simulation_results if res['team1_wins'] > res['team2_wins'])
    team2_wins = sum(1 for res in simulation_results if res['team2_wins'] > res['team1_wins'])
    conference_winners[conference] = teams[0] if team1_wins > team2_wins else teams[1]

# 결승 진출팀 결정
finals_teams = [conference_winners['west'], conference_winners['east']]
print(f"\n결승 진출팀: {finals_teams[0]} vs {finals_teams[1]}")

# 결승 데이터 필터링
valid_finals = valid[valid["TEAM_NAME"].isin(finals_teams)].copy()

# 결승 데이터 밸런싱
for team in finals_teams:
    valid_finals = balance_team_data(valid_finals, team, SIMULATION_CONFIG['min_games'])

# ------------------------
# [8] 결승전 시뮬레이션
# ------------------------
print("\n결승전 시뮬레이션 중...")
finals_simulation_results = []
for _ in tqdm(range(SIMULATION_CONFIG['n_simulations']), desc="결승전 시뮬레이션"):
    result = run_semifinal_simulation(
        finals_teams[0],
        finals_teams[1],
        valid_finals[valid_finals["TEAM_NAME"] == finals_teams[0]],
        valid_finals[valid_finals["TEAM_NAME"] == finals_teams[1]],
        features
    )
    finals_simulation_results.append(result)

# ------------------------
# [9] 결과 출력 및 저장
# ------------------------
print("\n=== NBA 4강 및 결승전 시뮬레이션 결과 ===")

# 4강 결과 출력
for conference, results in semifinal_results.items():
    print(f"\n{conference.upper()} 컨퍼런스 4강 결과:")
    team1, team2 = semifinal_teams[conference]
    
    team1_wins = sum(1 for res in results if res['team1_wins'] > res['team2_wins'])
    team2_wins = sum(1 for res in results if res['team2_wins'] > res['team1_wins'])
    team1_win_rate = team1_wins / SIMULATION_CONFIG['n_simulations'] * 100
    team2_win_rate = team2_wins / SIMULATION_CONFIG['n_simulations'] * 100
    
    print(f"\n승률:")
    print(f"{team1}: {team1_win_rate:.1f}%")
    print(f"{team2}: {team2_win_rate:.1f}%")
    
    # 평균 득점 계산
    team1_avg_score = np.mean([score['team1_score'] for res in results for score in res['series_scores']])
    team2_avg_score = np.mean([score['team2_score'] for res in results for score in res['series_scores']])
    team1_std_score = np.std([score['team1_score'] for res in results for score in res['series_scores']])
    team2_std_score = np.std([score['team2_score'] for res in results for score in res['series_scores']])
    
    print(f"\n평균 득점 (표준편차):")
    print(f"{team1}: {team1_avg_score:.1f} (±{team1_std_score:.1f})")
    print(f"{team2}: {team2_avg_score:.1f} (±{team2_std_score:.1f})")
    
    # 시리즈 스코어 분포
    series_scores = Counter([f"{res['team1_wins']}:{res['team2_wins']}" for res in results])
    print("\n시리즈 스코어 분포:")
    for score, count in series_scores.most_common():
        print(f"{score}: {count/SIMULATION_CONFIG['n_simulations']*100:.1f}%")
    
    # 가장 많이 나온 시리즈 스코어의 경기별 득점
    most_common_score = series_scores.most_common(1)[0][0]
    team1_wins, team2_wins = map(int, most_common_score.split(':'))
    total_games = team1_wins + team2_wins
    
    print(f"\n가장 많이 나온 시리즈 스코어 ({most_common_score})의 경기별 득점:")
    for game_num in range(total_games):
        game_scores = [score for res in results 
                      if f"{res['team1_wins']}:{res['team2_wins']}" == most_common_score 
                      for score in res['series_scores'] 
                      if score['game_num'] == game_num + 1]
        
        if game_scores:
            team1_avg = np.mean([score['team1_score'] for score in game_scores])
            team2_avg = np.mean([score['team2_score'] for score in game_scores])
            print(f"경기 {game_num + 1}: {team1} {team1_avg:.1f} - {team2} {team2_avg:.1f}")

# 결승전 결과 출력
print("\n=== 결승전 결과 ===")
team1, team2 = finals_teams

team1_wins = sum(1 for res in finals_simulation_results if res['team1_wins'] > res['team2_wins'])
team2_wins = sum(1 for res in finals_simulation_results if res['team2_wins'] > res['team1_wins'])
team1_win_rate = team1_wins / SIMULATION_CONFIG['n_simulations'] * 100
team2_win_rate = team2_wins / SIMULATION_CONFIG['n_simulations'] * 100

print(f"\n승률:")
print(f"{team1}: {team1_win_rate:.1f}%")
print(f"{team2}: {team2_win_rate:.1f}%")

# 평균 득점 계산
team1_avg_score = np.mean([score['team1_score'] for res in finals_simulation_results for score in res['series_scores']])
team2_avg_score = np.mean([score['team2_score'] for res in finals_simulation_results for score in res['series_scores']])
team1_std_score = np.std([score['team1_score'] for res in finals_simulation_results for score in res['series_scores']])
team2_std_score = np.std([score['team2_score'] for res in finals_simulation_results for score in res['series_scores']])

print(f"\n평균 득점 (표준편차):")
print(f"{team1}: {team1_avg_score:.1f} (±{team1_std_score:.1f})")
print(f"{team2}: {team2_avg_score:.1f} (±{team2_std_score:.1f})")

# 시리즈 스코어 분포
series_scores = Counter([f"{res['team1_wins']}:{res['team2_wins']}" for res in finals_simulation_results])
print("\n시리즈 스코어 분포:")
for score, count in series_scores.most_common():
    print(f"{score}: {count/SIMULATION_CONFIG['n_simulations']*100:.1f}%")

# 가장 많이 나온 시리즈 스코어의 경기별 득점
most_common_score = series_scores.most_common(1)[0][0]
team1_wins, team2_wins = map(int, most_common_score.split(':'))
total_games = team1_wins + team2_wins

print(f"\n가장 많이 나온 시리즈 스코어 ({most_common_score})의 경기별 득점:")
for game_num in range(total_games):
    game_scores = [score for res in finals_simulation_results 
                  if f"{res['team1_wins']}:{res['team2_wins']}" == most_common_score 
                  for score in res['series_scores'] 
                  if score['game_num'] == game_num + 1]
    
    if game_scores:
        team1_avg = np.mean([score['team1_score'] for score in game_scores])
        team2_avg = np.mean([score['team2_score'] for score in game_scores])
        print(f"경기 {game_num + 1}: {team1} {team1_avg:.1f} - {team2} {team2_avg:.1f}")

# 결과 저장
results_df = pd.DataFrame({
    'stage': ['east_semifinal', 'west_semifinal', 'finals'],
    'team1': [semifinal_teams['east'][0], semifinal_teams['west'][0], finals_teams[0]],
    'team2': [semifinal_teams['east'][1], semifinal_teams['west'][1], finals_teams[1]],
    'team1_win_rate': [team1_win_rate, team1_win_rate, team1_win_rate],
    'team2_win_rate': [team2_win_rate, team2_win_rate, team2_win_rate]
})

results_df.to_csv("C:/Users/THKIM/Desktop/3-1 프로젝트/오픈데이터분석 프로젝트/우승팀 예측/결승예측/nba_playoffs_simulation_results_4.csv", index=False, encoding="utf-8-sig")


[I 2025-05-27 22:09:17,366] A new study created in memory with name: no-name-eec47c65-ca68-4c09-9285-427fed1e8469



데이터 로드 중...

Optuna로 CatBoost 튜닝 중...


[I 2025-05-27 22:09:22,221] Trial 0 finished with value: 3.3388685829655937 and parameters: {'iterations': 489, 'depth': 4, 'learning_rate': 0.15963767403637047, 'l2_leaf_reg': 1.5948789542187933}. Best is trial 0 with value: 3.3388685829655937.
[I 2025-05-27 22:10:15,056] Trial 1 finished with value: 3.6852874346707423 and parameters: {'iterations': 752, 'depth': 9, 'learning_rate': 0.050657160951070646, 'l2_leaf_reg': 1.6986880346427862}. Best is trial 0 with value: 3.3388685829655937.
[I 2025-05-27 22:10:18,954] Trial 2 finished with value: 2.8033730077203765 and parameters: {'iterations': 436, 'depth': 5, 'learning_rate': 0.25322752768959783, 'l2_leaf_reg': 1.0233885623419994}. Best is trial 2 with value: 2.8033730077203765.
[I 2025-05-27 22:10:26,935] Trial 3 finished with value: 3.102679889603646 and parameters: {'iterations': 399, 'depth': 10, 'learning_rate': 0.24410600130948112, 'l2_leaf_reg': 4.242595471522881}. Best is trial 2 with value: 2.8033730077203765.
[I 2025-05-27 22

KeyboardInterrupt: 